# 광각(top) 재학습 — **wide_v3** (5클래스)

4종 도형 + `fruit_photo_cube` YOLOv8n. 광각 전용(본체 cube.pt와 별개).

- 업로드: **`dataset_wide_v3.zip`** (= 기존 wide 5class 데이터 + **wide 새 교정 라벨** 병합. 내부 루트 `wide_dataset/`). ← 새 wide 라벨 생기면 젯슨에서 만들어 줌.
- `yolov8n.pt`서 새로. best.pt → 젯슨 `models/wide.pt` 배포(광각용, cube.pt는 본체 유지).
- **imgsz=1280**(런타임 일치). ⚠️ **run명 `wide_v3`**(이전 wide_v2와 구분).
- 런타임 → GPU 켜고 실행.

In [ ]:
# [셀1] 업로드 + 압축해제 — dataset_wide_v3.zip (내부 루트 wide_dataset/)
from google.colab import files
import os, shutil
up = files.upload()
ZIP = next(iter(up))
shutil.rmtree('/content/wide_dataset', ignore_errors=True)
!unzip -o -q "$ZIP" -d /content
print('uploaded:', ZIP, '-> extracted:', sorted(os.listdir('/content/wide_dataset')))

In [ ]:
# [셀2] train/val 분리 + 코랩용 data.yaml 생성 (nc=5, fruit_photo_cube 포함)
import os, glob, random, shutil, yaml
random.seed(0)
ROOT = '/content/wide_dataset'
HELD_OUT_VAL = True      # ← False면 train==val (전체 학습, 검증은 live로)
VAL_FRAC = 0.15
lbl = lambda p: f"/content/wide_dataset/labels/" + os.path.splitext(os.path.basename(p))[0] + ".txt"
pairs = [(i, lbl(i)) for i in sorted(glob.glob(f'/content/wide_dataset/images/*')) if os.path.exists(lbl(i))]
print('pairs:', len(pairs), '(빈 라벨=배경음성 포함)')
if HELD_OUT_VAL:
    random.shuffle(pairs); n = int(len(pairs) * VAL_FRAC)
    for split, items in [('train', pairs[n:]), ('val', pairs[:n])]:
        for s in ('images', 'labels'):
            os.makedirs(f'/content/wide_dataset/{split}/{s}', exist_ok=True)
        for img, lb in items:
            shutil.copy(img, f'/content/wide_dataset/{split}/images/'); shutil.copy(lb, f'/content/wide_dataset/{split}/labels/')
    print('train', len(pairs) - n, '/ val', n)
    data = dict(path=ROOT, train='train/images', val='val/images')
else:
    data = dict(path=ROOT, train='images', val='images')
data.update(nc=5, names=['cube', 'octahedron', 'dodecahedron', 'icosahedron', 'fruit_photo_cube'])
yaml.safe_dump(data, open(f'/content/wide_dataset/data_colab.yaml', 'w'))
print(open(f'/content/wide_dataset/data_colab.yaml').read())

In [ ]:
# [셀3] 학습 — 광각 전용, yolov8n서 새로. imgsz=1280. run명 wide_v3 (버전 구분)
!pip -q install ultralytics
from ultralytics import YOLO
YOLO('yolov8n.pt').train(data='/content/wide_dataset/data_colab.yaml',
    epochs=100, imgsz=1280, batch=8, patience=30, name='wide_v3')

In [ ]:
# [셀4] best.pt 내려받기 → 젯슨 models/wide.pt 배포(광각용)
from google.colab import files
files.download('runs/detect/wide_v3/weights/best.pt')